In [1]:
!pip install pyspark pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 3.3 MB/s eta 0:00:00a 0:00:01


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CatsDogsExample")
    .getOrCreate()
)

In [3]:
from pyspark.sql.functions import col, regexp_extract, when

# 1) Lecture des images
images_df = (
    spark.read
         .format("image")              # data source "image" de Spark
         .option("dropInvalid", True)
         .load("../data/cats_dogs/*/*")   # sous-dossiers cats/ et dogs/
)

images_df.printSchema()
images_df.select("image.origin", "image.width", "image.height").show(5, truncate=False)

root
 |-- image: struct (nullable = true)
 |    |-- origin: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |    |-- nChannels: integer (nullable = true)
 |    |-- mode: integer (nullable = true)
 |    |-- data: binary (nullable = true)

+-----------------------------------------------------------------------------------------------------+-----+------+
|origin                                                                                               |width|height|
+-----------------------------------------------------------------------------------------------------+-----+------+
|file:///home/jovyan/data/cats_dogs/cats/Bombay_140_jpg.rf.15757f698af74453f34ee58604ea6afc.jpg       |1725 |1600  |
|file:///home/jovyan/data/cats_dogs/cats/Maine_Coon_10_jpg.rf.7197163a4c9374ee5e54107828067bd5.jpg    |600  |900   |
|file:///home/jovyan/data/cats_dogs/dogs/shiba_inu_183_jpg.rf.0ab5e35c66b37de6d32c61d00e9ce63c.jpg    |500  |460

In [4]:
from pyspark.sql.functions import regexp_extract

# Extraire "cats" ou "dogs" depuis le chemin du fichier
images_labeled = images_df.withColumn(
    "label_str",
    regexp_extract(col("image.origin"), r"/(cats|dogs)/[^/]+$", 1)
)

# On garde seulement les lignes correctement labelisées
images_labeled = images_labeled.filter(col("label_str") != "")

# Convertir en label numérique (0 = chat, 1 = chien par exemple)
images_labeled = images_labeled.withColumn(
    "label",
    when(col("label_str") == "cats", 0.0).otherwise(1.0)
)

images_labeled.select("image.origin", "label_str", "label").show(10, truncate=False)

+-----------------------------------------------------------------------------------------------------------------+---------+-----+
|origin                                                                                                           |label_str|label|
+-----------------------------------------------------------------------------------------------------------------+---------+-----+
|file:///home/jovyan/data/cats_dogs/cats/Bombay_140_jpg.rf.15757f698af74453f34ee58604ea6afc.jpg                   |cats     |0.0  |
|file:///home/jovyan/data/cats_dogs/cats/Maine_Coon_10_jpg.rf.7197163a4c9374ee5e54107828067bd5.jpg                |cats     |0.0  |
|file:///home/jovyan/data/cats_dogs/dogs/shiba_inu_183_jpg.rf.0ab5e35c66b37de6d32c61d00e9ce63c.jpg                |dogs     |1.0  |
|file:///home/jovyan/data/cats_dogs/dogs/japanese_chin_137_jpg.rf.85a9cb6d5086cd4d97755c7d4e87ba48.jpg            |dogs     |1.0  |
|file:///home/jovyan/data/cats_dogs/dogs/havanese_157_jpg.rf.18088a3ef4a667e

In [5]:
import numpy as np
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.ml.linalg import Vectors, VectorUDT

# UDF qui transforme l'image (struct) en liste de floats
def image_to_vector(image):
    # image.data : bytes BGR (voir doc Spark ImageSchema) :contentReference[oaicite:2]{index=2}
    if image is None:
        return None
    
    # On reconstruit un tableau numpy (H, W, C)
    h = image.height
    w = image.width
    c = image.nChannels
    arr = np.frombuffer(image.data, dtype=np.uint8)
    if arr.size != h * w * c:
        return None

    arr = arr.reshape((h, w, c))
    
    # Passage en niveaux de gris simple : moyenne sur les canaux
    gray = arr.mean(axis=2)

    # Normalisation 0–1
    gray = gray / 255.0

    # Flatten
    flat = gray.flatten().astype("float32")

    return flat.tolist()

image_to_vec_udf = udf(image_to_vector, ArrayType(DoubleType()))

images_feat = images_labeled.withColumn("features_array", image_to_vec_udf(col("image")))
images_feat.select("image.origin", "label", "features_array").show(2, truncate=True)

+--------------------+-----+--------------------+
|              origin|label|      features_array|
+--------------------+-----+--------------------+
|file:///home/jovy...|  0.0|[0.61960786581039...|
|file:///home/jovy...|  0.0|[0.16993464529514...|
+--------------------+-----+--------------------+
only showing top 2 rows



In [6]:
import numpy as np
from PIL import Image   # pip install pillow
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.sql.functions import udf, col

TARGET_W = 100
TARGET_H = 100

def image_to_vector_resize(image):
    if image is None:
        return None

    h = image.height
    w = image.width
    c = image.nChannels

    arr = np.frombuffer(image.data, dtype=np.uint8)
    if arr.size != h * w * c:
        return None

    arr = arr.reshape((h, w, c))

    # Spark stocke souvent en BGR : on passe en RGB (optionnel selon ton cas)
    if c >= 3:
        rgb = arr[:, :, :3][:, :, ::-1]  # BGR -> RGB
    else:
        rgb = np.stack([arr[:, :, 0]] * 3, axis=2)

    # Conversion en image PIL + resize
    im = Image.fromarray(rgb.astype("uint8"), mode="RGB")
    im = im.resize((TARGET_W, TARGET_H))

    # Niveau de gris
    gray = np.array(im).mean(axis=2) / 255.0

    flat = gray.flatten().astype("float32")
    return flat.tolist()

image_to_vec_udf = udf(image_to_vector_resize, ArrayType(DoubleType()))

images_feat = images_labeled.withColumn("features_array", image_to_vec_udf(col("image")))


In [7]:
from pyspark.ml.linalg import Vectors, VectorUDT

to_vector_udf = udf(lambda xs: Vectors.dense(xs) if xs is not None else None, VectorUDT())
data = images_feat.withColumn("features", to_vector_udf(col("features_array"))).drop("features_array")
data = data.filter(col("features").isNotNull())

In [8]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Split train / test
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

pipeline = Pipeline(stages=[lr])
model = pipeline.fit(train_df)

# Évaluation
pred = model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

roc = evaluator.evaluate(pred)
print("AUC ROC =", roc)

pred.select("image.origin", "label", "prediction", "probability").show(10, truncate=False)

AUC ROC = 0.5195652173913043
+-----------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|origin                                                                                                           |label|prediction|probability                               |
+-----------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|file:///home/jovyan/data/cats_dogs/cats/Maine_Coon_10_jpg.rf.7197163a4c9374ee5e54107828067bd5.jpg                |0.0  |0.0       |[0.9999998039695548,1.9603044520977164E-7]|
|file:///home/jovyan/data/cats_dogs/dogs/english_cocker_spaniel_142_jpg.rf.1f40aa6bf596bc8c4b33ded80200848b.jpg   |1.0  |1.0       |[3.096403722899549E-18,1.0]               |
|file:///home/jovyan/data/cats_dogs/dogs/great_pyrenees_148_jpg.rf.6ed65570f2b5988c274ca94d

In [9]:
model.write().overwrite().save("models/my_model")